# Model C: TACR (Transformer Actor-Critic with Regularization)

Phase 3, Model C of the four-model comparison. Reproduction of Lee & Moon,
"Transformer Actor-Critic with Regularization: Automated Stock Trading using
Reinforcement Learning", IEEE Access 2023 (DOI 10.1109/ACCESS.2023.3324458;
authors' code: github.com/VarML/TACR).

**What this notebook does**
1. Trains the TACR actor+critic offline on the four behavior-policy
   trajectories (Phase-1 offline dataset; time-split train/val/test).
2. Shows training curves (actor BC+Q loss, critic TD loss, val Sharpe).
3. Runs the basin-screening check for the seed (best-val epoch + cross-seed
   correlation; reseed rule from Model B).
4. Rolls the test set and reports the regime breakdown vs Model B (naive_new)
   and the equal-weight-momentum (EM) control.

Deviations from the paper are flagged in `src/models/tacr/` docstrings and
`configs/tacr.yaml`; the canonical 5-seed results live in the eval CLI output
(`python -m src.models.tacr.eval` -> checkpoints/tacr/*.csv).


In [ ]:
import sys
from pathlib import Path

# notebooks/ or repo root depending on kernel cwd
root = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from src.models.tacr.config import TACRConfig
from src.models.tacr.data import load_tacr_data
from src.models.tacr.train import train_tacr
from src.models.tacr.eval import roll_test_predictions, exposure_and_turnover

SEEDS = [20260814, 1, 2, 3, 4]
cfg = TACRConfig.from_yaml()  # configs/tacr.yaml (paper values + flagged deviations)
print(cfg)

In [ ]:
# single-seed run for inspection (the CLI trains all SEEDS with the same
# protocol; seeds train independently, so re-running here is equivalent)
cfg.seed = SEEDS[0]
cfg.checkpoint_dir = cfg.checkpoint_dir / f"s{cfg.seed}"
data = load_tacr_data(cfg.u)
model, history, best_path = train_tacr(cfg, data)
history

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(history["epoch"], history["actor_loss"], marker="o")
axes[0].set_title("actor loss (-lambda*Q + BC)")
axes[1].plot(history["epoch"], history["critic_loss"], marker="o")
axes[1].set_title("critic TD loss")
axes[2].plot(history["epoch"], history["val_sharpe"], marker="o")
axes[2].set_title("val Sharpe (annualized)")
axes[2].axhline(0, color="gray", lw=0.5)
plt.tight_layout()

**Basin screening** (protocol carried from Model B): a good-basin run peaks
early in the schedule (Model B pack: best-val at epochs 3-5 of 30; the bad
basin at 29/30). Flag if best-val epoch > 35% of the schedule, or if the
seed's test actions correlate < 0.7 with the pack. Flagged seeds are
re-seeded, never included. The full 5-seed screening table is written by
`python -m src.models.tacr.eval` to checkpoints/tacr/basin_screening.csv.

In [ ]:
log = history
best_ep = int(log["val_sharpe"].idxmax()) + 1
print(f"seed {cfg.seed}: best-val epoch {best_ep} of {len(log)} "
      f"(flag > {round(0.35 * len(log))})")
print("-> ok" if best_ep <= round(0.35 * len(log)) else "-> FLAG: suspect basin, reseed")

In [ ]:
# test roll for this seed (constant RTG 0.0; own autoregressive actions)
preds = roll_test_predictions(cfg, checkpoint=best_path, data=data)
preds

In [ ]:
# regime breakdown for this seed; the CLI aggregates all 5 seeds
import numpy as np
import pandas as pd
import src.eval.regime_eval as re

rows = []
expo = exposure_and_turnover(preds)
for regime in ("bull", "bear", "crisis", "all"):
    mask = (preds["regime"] == regime) if regime != "all" else np.ones(len(preds), bool)
    r = preds["ret"].to_numpy()[mask]
    if len(r) < 2:
        continue
    rec = {"regime": regime, "sharpe": round(re.sharpe_ratio(r), 4),
           "mean_action": round(float(preds["action"].to_numpy()[mask].mean()), 4),
           "turnover": round(expo["turnover"], 4)}
    rows.append(rec)
pd.DataFrame(rows)

**Reading the table**

- Per-regime Sharpe is the PRIMARY result; `all` (blended) is secondary.
- `turnover` = mean |a_t - a_{t-1}| over test days (log stability proxy).
- Crisis regime: 15 test days — directionally suggestive only, NOT a finding.
- The cross-seed comparison vs Model B (naive_new) and EM lives in
  `checkpoints/tacr/regime_eval.csv` / `vs_model_b.csv` from the eval CLI
  (same 5-seed protocol, wins criterion >= 3/5 seeds).